# current ds - center is N

In [1]:
# imorts

# imports

# ml
import torch
import torch.nn as nn
from torchvision.models import vgg16
import torch.optim.lr_scheduler as lr_scheduler
import torch.optim as optim
from torchvision.models import vgg16
from torch.utils.data import DataLoader
#from torch.Utils.data import DataLoader
import torch.nn.functional as F

from sklearn.model_selection import train_test_split

# general maths and image manipulation
import numpy as np
import cv2

# general other
from datetime import date
from tqdm import tqdm
import pprint
import collections
from IPython.display import clear_output
import time
import random


# saving in file types
import csv
import json
import pickle
import os

# simulation logger
import wandb

# my functions
import sys
sys.path.append('../../.')
from functions import  ImageProcessor
from dataPreProcessingP3Direction import get_data
from dataloaderP3Direction import IDSWDataSetLoader3
from fns4wandb import set_lossfn
from architectures import eightnnet, PrintLayer
from loopsP3Direction import loop_batch, test_loop_batch, train_val_batch
from plotting import learning_curve, accuracy_curve
from plottingP3Direction import plot_confusion
from modelCardsP3Direction import Cards, get_lin_lay, return_card
from modelManagment import choose_model,choose_scheduler

from fileManagment import save2csv_nest_dict, check_obj4np, save2josn_nested_dict, save2csv, save2json,read_in_json


/its/home/nn268/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
/its/home/nn268/.local/lib/python3.10/site-packages/pandas/core/arrays/masked.py:62: UserWarning: Pandas requires version '1.3.4' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


In [2]:
# directories 

save_location  = "/its/home/nn268/antvis/antvis/optics/res_big_loop_saves/models/p3/testing/" # "/its/home/nn268/antvis/antvis/optics/res_big_loop_saves//models/p3/DIRECTIONLEARNING/"

data_path = "/its/home/nn268/antvis/antvis/optics/NC_IDSW/"

gitHASH = " 32d139c134ef0530871dc082bae2923877f71936"

model_name = '6c3l'

epochs = 300

tv = [452, 144]

projectNAME = f"RealThing__{model_name}_300E_1e-4_ADAM_Eighths_{tv}"#f"LargeModels_ALLLOCS_{model_name}_{tv}_{epochs}E_1e-4_FixedPeakDistERR" #"test"#

full_path = save_location+projectNAME+"/"+model_name+"/"
if not os.path.exists(full_path):
    os.makedirs(full_path)

save_location = full_path

In [3]:
device = "cuda:1" if torch.cuda.is_available() else "cpu"
print(device)

cuda:1


In [4]:
len(os.listdir(data_path)) # 2744

2745

In [5]:
# wandb log

wandb.login()

wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /its/home/nn268/.netrc.
wandb: Currently logged in as: naughticalnonsence (antvis) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [6]:
# import model cards
cards = Cards()
modelcards = cards.modelcards

if model_name != 'resnet18':
    modelcard = return_card(modelcards, key='name', targetValue=model_name)
    print(modelcard)
resolutioncards = cards.resolutioncards
resolutioncard = return_card(resolutioncards, key='resolution', targetValue=tv)



[{'name': '6c3l', 'model': '6c3l', 'channels': 3, 'Ks': (3, 5), 'f_lin_lay': [267520, 66560, 15360, 3840, 1024, 193024, 193024], 'idx': 3, 'dropout': 0.2}]




modelcard[0]['f_lin_lay'][0] = 1032192    ## Just for testing an individual model - Remove and make new modelcards3PDirections file in real.

print(modelcard)
print(resolutioncard)

In [7]:
# create config dict - add cards and parameters

learning_rate =  1e-4


seeds = [42, 7, 56, 23, 22, 69]


batchsize = 64


half_ciprange = 22 # (roughly half of 45)
std_dev = 7



loss_fn = ['MSE']
optim = ["adam"]

scheduler_value = "NoSched"

config = dict({'name': f"Sweep on {model_name} at {resolutioncard[0]['resolution']}_clip:{half_ciprange}"}) #config = dict({'name': 'Sweep on 2C'})


config.update({"method": "bayes", "metric":{"goal": "minimize", "name": "t_loss"},
              "parameters": {"epochs" :{"value" : epochs},
                             "batch_size": {"value": batchsize},
                             "learning_rate":{"value": learning_rate},
                             "loss_fn_cards": {"value":loss_fn},
                             "optimiser": {"value": optim},
                             "seeds": {"values": seeds},
                              "half_ciprange": {"value":half_ciprange},
                             "std_dev":{"value":std_dev},
                             'model_cards':{"value":modelcard},
                             'resolution_cards':{"value":resolutioncard}
                            }})


In [8]:
def getAcc_fromdict(listdict):
    baseacc = []
    MSE = []
    MAE = []
    peak = []
    for item in listdict:
        baseacc.append(item['baseAcc'])
        MSE.append(item['MSE'])
        MAE.append(item['MAE'])
        peak.append(item['peakDist'])
    return baseacc, MSE, MAE, peak

In [9]:

def _go(config=None):
    
    if len(gitHASH) <1:
        print("YOU FORGET THE GIT HASH")
        return
    else:
        #print('Git Hash registered')
        pass
    d = date.today()
    mc = modelcard[0]
    with wandb.init(config=config):  
        config = wandb.config

        for model_idx, model_card in enumerate(modelcard): #(config['model_cards']):
            start = time.process_time()
            print(model_card)
            model_name = model_card['model']
            model_index = model_card['idx']
            dropout = model_card['dropout'] 
            
            output_lin_lay = 360 ###### Output labels for direction prediction specifically. 
            
            
            for res_idx, resolution_card in enumerate(resolutioncard):
                resolution = resolution_card['resolution']
                lin_lay = get_lin_lay(model_card, resolution)
            
                seed = config.seeds
                loss = config.loss_fn_cards
                torch.cuda.empty_cache()

                batch = config.batch_size 

                print('Model: ', str(model_name), f" idx: {model_idx} / {len(config.model_cards)}")
                print('resolution: ', str(resolution), f" idx: {res_idx} / {len(config['resolution_cards'])}")
                print('seed: ', str(seed))
                print('loss function: ', str(loss))
                print('Batch size: ', config.batch_size)
                print('Training epochs: ', config.epochs)
                print(device)
                run_start_time = time.process_time()
                print('start time: ',run_start_time)

                epochs = config['epochs'] #40

                IP = ImageProcessor(device)

                wandb.log({'gitHash':gitHASH})
                wandb.log({'Epochs': epochs})
                wandb.log({'schedType':scheduler_value})

                save_dict = {'Run' : f"{model_name}_{resolution}_{d}",
                             'start_epoch' : 0,
                             'Current_Epoch': 0,
                             'save_location' : save_location,
                            'scheduler': scheduler_value}

                if model_name == 'resnet18':
                    model = resnet18(weights=None, num_classes =360).to(device)
                    model_index = 100
                else:
                    lin_lay = get_lin_lay(model_card, resolution)
                    dropout = model_card['dropout'] 
                    print(f"Model Name {model_name}")
                    model = choose_model(model_name, lin_lay, dropout, output_lin_lay).to(device)

                half_ciprange = config.half_ciprange
                std_dev = config.std_dev

                x_train, y_train, x_val, y_val, x_test, y_test = get_data(seed, "/its/home/nn268/antvis/antvis/optics/NC_IDSW/")
                av_lum = IP.new_luminance(x_train)
                train = (x_train, resolution, av_lum, model_name, config.half_ciprange, config.std_dev, batch)
              
                # FOR increases DS size in training via augmentations (yaw augmentations) only create the DSL for test here, Train and Val in epoch loop
                # that will give different yaw augmentations each loop
                # if i also increase epochs, I get more unique tries for direction learning
                
                test_ds= IDSWDataSetLoader3(x_test, resolution,av_lum,model_name, config.half_ciprange, config.std_dev, device)
                test = DataLoader(test_ds, batch_size=config.batch_size, shuffle=True, drop_last=True) #, num_workers=2
    
                #print(f"len training : {len(x_train)}     len val : {len(x_val)}    len test : {len(x_test)}")

                loss_fn = set_lossfn(loss)
                
                # set optimizer
                optimizer = torch.optim.Adam(model.parameters(),lr=config.learning_rate)
                scheduler = choose_scheduler(save_dict, optimizer)

                wandb.watch(model, loss_fn, log='all', log_freq=2, idx = model_index)

                loop_run_name = f"{save_dict['Run']}_{resolution}_{config.learning_rate}_{scheduler_value}_{seed}_{loss}"
                model, save_dict = train_val_batch(model, train, x_val, save_dict, config.learning_rate, loss_fn,epochs, config.batch_size, optimizer, scheduler_value, device, config)

                test_acc, test_predict_list, y_test = test_loop_batch(model,test, loss_fn, config.batch_size, device,config, runname=loop_run_name, save_loc = save_dict['save_location']) #model, model_name, X, Y, res, pad, loss_fn, device, num_classes=11
                test_predict_numerical = [p.item() for p in test_predict_list]
                y_test_numerical = [y.item() for y in y_test]
                print(np.unique(test_predict_list))
                
                print(' \n train Acc: ', save_dict['t_accuracy_list'][-1])
                print(' \n val Acc: ', save_dict['v_accuracy_list'][-1])
                print(' \n test Acc: ', test_acc)
                
                save_dict.update({'test_acc': test_acc})
                save_dict.update({'test_predict': test_predict_list})
                save_dict.update({'test_labels': list(y_test)})

                t_acc = save_dict['t_accuracy_list']
                tbase_acc, tMSE, tMAE, t_peakdist = getAcc_fromdict(t_acc)
                
                v_acc = save_dict['v_accuracy_list']
                vbase_acc, vMSE, vMAE, v_peakdist = getAcc_fromdict(v_acc)

                learning_curve(save_dict['t_loss_list'], save_dict['v_loss_list'], save_location=save_dict['save_location'],run_name=loop_run_name)
                
                accuracy_curve(tbase_acc, vbase_acc ,save_location=save_dict['save_location'],run_name="Basic"+loop_run_name)
                accuracy_curve(tMSE, vMSE ,save_location=save_dict['save_location'],run_name="MSE"+loop_run_name)
                accuracy_curve(tMAE, vMAE ,save_location=save_dict['save_location'],run_name="MAE"+loop_run_name)
                accuracy_curve(t_peakdist, v_peakdist ,save_location=save_dict['save_location'],run_name="PeakDist"+loop_run_name)
                
                #print(f"test_predict_numerical :   {test_predict_numerical}")
                #print(f"y_test_numerical :     {y_test_numerical}")
                plot_confusion(predictions= test_predict_numerical, actual= y_test_numerical, title = "Test Confusion matrix", run_name = loop_run_name,save_location =save_dict['save_location'])
                
                
                wandb.log({'test_predict': test_predict_list})
                wandb.log({'test_labels': list(y_test)})
                #saving
                diction = {}
                d = date.today()
                d=str(d)
                diction.update({'Date':d})
                diction.update({'gitHASH':str(gitHASH)})
                diction.update({'model_name': str(model_name)})
                diction.update({'loss_fn': str(loss)})
                diction.update({'lr': str(config.learning_rate)})
                diction.update({'seed': str(seed)})
                diction.update({'resolution': str(resolution)})
                diction.update({'lin_lay': int(lin_lay)})
                diction.update({'run time': (time.process_time() - run_start_time)})
                diction.update(save_dict)
                
                _save_location = save_dict['save_location']
                title = save_dict['Run']
                save2json(diction, loop_run_name, _save_location)
                save2csv(diction, title, _save_location)

                diction['model.state_dict'] = model.state_dict() #to('cpu').

                with open(f"{save_location}{loop_run_name}.pkl", 'wb+') as f:
                    pickle.dump(diction, f)
                                
                print(f' \n END {model_name} {resolution} Run Time: ',time.process_time() - run_start_time)
                torch.cuda.empty_cache()


In [10]:
sweep_id = wandb.sweep(sweep= config, project=projectNAME)

wandb.agent(sweep_id, function=_go, count=20)

Create sweep with ID: gprke4v5
Sweep URL: https://wandb.ai/antvis/RealThing__6c3l_300E_1e-4_ADAM_Eighths_%5B452%2C%20144%5D/sweeps/gprke4v5


wandb: Agent Starting Run: 2owc2ex7 with config:
wandb: 	batch_size: 64
wandb: 	epochs: 300
wandb: 	half_ciprange: 22
wandb: 	learning_rate: 0.0001
wandb: 	loss_fn_cards: ['MSE']
wandb: 	model_cards: [{'Ks': [3, 5], 'channels': 3, 'dropout': 0.2, 'f_lin_lay': [267520, 66560, 15360, 3840, 1024, 193024, 193024], 'idx': 3, 'model': '6c3l', 'name': '6c3l'}]
wandb: 	optimiser: ['adam']
wandb: 	resolution_cards: [{'index': 0, 'padding': 5, 'resolution': [452, 144]}]
wandb: 	seeds: 56
wandb: 	std_dev: 7
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /its/home/nn268/.netrc.


{'name': '6c3l', 'model': '6c3l', 'channels': 3, 'Ks': (3, 5), 'f_lin_lay': [267520, 66560, 15360, 3840, 1024, 193024, 193024], 'idx': 3, 'dropout': 0.2}
Model:  6c3l  idx: 0 / 1
resolution:  [452, 144]  idx: 0 / 1
seed:  56
loss function:  ['MSE']
Batch size:  64
Training epochs:  300
cuda:1
start time:  12.360255209
Model Name 6c3l
/its/home/nn268/antvis/antvis/optics/NC_IDSW/
set_lossfn    lf  ['MSE']


  0%|                               | 0/300 [00:00<?, ?it/s]

Data Loading...
Training...
Optimizer present:  Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0001
    maximize: False
    weight_decay: 0
)


...

  0%|                               | 0/300 [00:00<?, ?it/s]

Data Loading...
Training...
Optimizer present:  Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0001
    maximize: False
    weight_decay: 0
)



Traceback (most recent call last):
  File "/tmp/ipykernel_220648/2081651180.py", line 92, in _go
    model, save_dict = train_val_batch(model, train, x_val, save_dict, config.learning_rate, loss_fn,epochs, config.batch_size, optimizer, scheduler_value, device, config)
  File "/its/home/nn268/antvis/antvis/optics/Batchcode/p3/../.././loopsP3Direction.py", line 327, in train_val_batch
    t_loss, train_prediction, t_label_list, tacc, model, optimizer, img_batch, imNorm_batch = loop_batch(model,
  File "/its/home/nn268/antvis/antvis/optics/Batchcode/p3/../.././loopsP3Direction.py", line 149, in loop_batch
    prediction = model.forward(x_batch.to(device))
  File "/its/home/nn268/antvis/antvis/optics/Batchcode/p3/../.././architectures.py", line 105, in forward
    x= self.conv_layers(x)
  File "/its/home/nn268/.local/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1518, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "/its/home/nn268/.local/lib/pyth

Epochs,▁
Epochs,300
gitHash,32d139c134ef0530871...
schedType,NoSched


Traceback (most recent call last):
  File "/its/home/nn268/.local/lib/python3.10/site-packages/wandb/agents/pyagent.py", line 296, in _run_job
    self._function()
  File "/tmp/ipykernel_220648/2081651180.py", line 92, in _go
    model, save_dict = train_val_batch(model, train, x_val, save_dict, config.learning_rate, loss_fn,epochs, config.batch_size, optimizer, scheduler_value, device, config)
  File "/its/home/nn268/antvis/antvis/optics/Batchcode/p3/../.././loopsP3Direction.py", line 327, in train_val_batch
    t_loss, train_prediction, t_label_list, tacc, model, optimizer, img_batch, imNorm_batch = loop_batch(model,
  File "/its/home/nn268/antvis/antvis/optics/Batchcode/p3/../.././loopsP3Direction.py", line 149, in loop_batch
    prediction = model.forward(x_batch.to(device))
  File "/its/home/nn268/antvis/antvis/optics/Batchcode/p3/../.././architectures.py", line 105, in forward
    x= self.conv_layers(x)
  File "/its/home/nn268/.local/lib/python3.10/site-packages/torch/nn/modules/

In [11]:
#_go(config)